# Intro

In [40]:
import os, sys
import numpy as np
import pickle 

In [67]:
from collections import defaultdict


In [41]:
from dcms.models import DCMModel, DECMModel, ADECMModel, DWCMModel

## Load data

In [42]:
HOME='/home/sarawalk/bowtie2_py39/bowtie2/'

In [43]:
sys.path.insert(0, HOME)

In [44]:
DATA_FOLDER=HOME+'dati_elezioni/'

In [45]:
files=os.listdir(DATA_FOLDER)
files.sort()

In [46]:
files

['crisi_dicos.csv',
 'crisi_weighted_edgelist.csv',
 'ita_elections_dicos.csv',
 'ita_elections_weighted_edgelist.csv',
 'quirinale_dicos.csv',
 'quirinale_weighted_edgelist.csv']

In [47]:
l_dataset=len(files)//2

## Functions

In [48]:
def el2ks(el):
    all_nodes=np.concatenate((el['source_id'], el['target_id']))
    all_nodes=np.unique(all_nodes)
    k_out=np.zeros(len(all_nodes), dtype=int)
    k_in=np.zeros(len(all_nodes), dtype=int)
    s_out=np.zeros(len(all_nodes), dtype=int)
    s_in=np.zeros(len(all_nodes), dtype=int)
    node_index={node:i for i,node in enumerate(all_nodes)}
    for s,t,w in el:
        i_s=node_index[s]
        i_t=node_index[t]
        k_out[i_s]+=1
        k_in[i_t]+=1
        s_out[i_s]+=w
        s_in[i_t]+=w
    return k_out, k_in, s_out, s_in, all_nodes

# Case0: Crisis

In [49]:
i_file=0

In [117]:
el=np.genfromtxt(
    DATA_FOLDER+files[i_file+1],
    delimiter=',',
    skip_header=1,
    autostrip=True,
    dtype=[('source_id', '>U50'), ('target_id', '>U20'),('weight', 'i4')]
 )
#el['source_id'] = np.char.strip(el['source_id'])
#l['target_id'] = np.char.strip(el['target_id'])

In [119]:
dico=np.genfromtxt(
    DATA_FOLDER+files[i_file],
    delimiter=',',
    skip_header=1,
    autostrip=True,
    dtype=[('user_id', '>U50'), ('dico', '>U2'), ('h_dico', 'U2'), ('i_dico', 'U2')]
 )
#dico['user_id'] = np.char.strip(dico['user_id'])
#dico['dico'] = np.char.strip(dico['dico'])

In [123]:
dico_dict={}
bad_dicos=[]
for d in dico:
    if d['dico'].isnumeric():
        dico_dict[d['user_id']]=int(d['dico'])
    else:
        if d['dico'] not in bad_dicos:
            bad_dicos.append(d['dico'])
            print(d['dico'])

-1


In [124]:
np.unique(list(dico_dict.values()), return_counts=True)

(array([0, 1, 2, 3, 4]), array([58832, 31874, 15168,  1304,  3637]))

In [125]:
len(aux[4]), len(dico_dict), (len(aux[4])-len(dico_dict))/len(aux[4])

(130666, 110815, 0.15192169347802795)

Who are the missing ones?

In [126]:
# Raggruppo in liste per efficienza, poi converto in array strutturati come el
_tmp = defaultdict(list)
for edge in el:
    src = edge['source_id'].strip()
    tgt = edge['target_id'].strip()
    d_src = dico_dict.get(src)
    if d_src is not None and d_src == dico_dict.get(tgt):
        _tmp[d_src].append(edge)

# defaultdict che restituisce array vuoti con lo stesso dtype di el
el_dico = defaultdict(
    lambda: np.empty(0, dtype=el.dtype),
    {k: np.array(v, dtype=el.dtype) for k, v in _tmp.items()}
 )

In [129]:
for key in el_dico.keys():
    print(key, len(el_dico[key]))

1 345876
0 322098
2 143937
3 1748
4 4578


In [130]:
del _tmp

## From the edgelist to the constraints

#### DiCo 1

In [131]:
aux=el2ks(el_dico[1])# DiCo 1

In [133]:
assert aux[0].sum()==aux[1].sum()==len(el_dico[1])

In [134]:
assert aux[2].sum()==aux[3].sum()

In [141]:
# Number of nodes, Number of edges, edge density
len(aux[4]), len(el_dico[1]), len(el_dico[1])/len(aux[4])**2

(31874, 345876, 0.0003404452594366783)

Ok, that's really challenging. So far, DCM Models were tested on datasets with 5k nodes. Here we have a little less than 32k nodes, and the number of links is a little more than 345k. Let's see if it works.

## DCM Models

### DWCM

In [142]:
dwcm=DWCMModel(aux[2], aux[3])

In [143]:
dwcm.solve_tool(tol=1e-5, max_iter=10000)

Converged in 27 iteration(s).


True